# Calibration mocks for the fits (not part of the pipeline)

For each sample: Gaussian mocks drawn from the sample's own best-fit model and covariance,
refit with the same settings. Checks that the reported errors are honest:
* **pull std** of $(\hat\alpha - \alpha_{\rm true})/\sigma_\alpha$ should be ≈ 1;
* **68 % coverage** of the error intervals should be ≈ 0.68 (Δχ² = 1 and marginal);
* for Σ: bias and scatter of the marginal mean, coverage, and how often the Δχ² = 1 interval is open.

These validate the fitting machinery (grids, priors, error definitions), not the covariance,
because the mocks are drawn from it. Run once per set of results for the paper.

In [ ]:
import os, sys
REPO = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.insert(0, os.path.join(REPO, 'src'))
from bao_fitting import run_all, run_mocks

MONO = os.path.join(REPO, 'data', 'processed', 'monopoles', 'box')
PK = os.path.join(REPO, 'data', 'processed', 'pk', 'box')

In [ ]:
# Same samples and settings as in notebooks/final/bao_fitting.ipynb
binsep = 5.0
base = os.path.join(MONO, f'mag=-21.2_sep=5.0-150.0_binsep={binsep:.1f}_')
samples = {
    'Full':         base + 'full.npz',
    'Filament':     base + '0.0-d_mathrmfilleq5.0.npz',
    'Non-filament': base + '10.0-d_mathrmfilleq100.0.npz',
}
pk_file = os.path.join(PK, 'pk_XISMU_box_mag-21.2_dist_fil.npz')
res = run_all(samples, pk_file, fit_range=(65, 135), n_mock=100, plot=False)

In [ ]:
n_mock = 300
for lab, r in res.items():
    mk = run_mocks(r['fit'], n_mock=n_mock, seed=1, truth='best')            # alpha fit
    mkS = run_mocks(r['fit_free'], n_mock=n_mock, seed=3, truth='best')      # Sigma fit
    print(f"{lab}")
    print(f"  alpha fit: bias {mk['bias']:+.4f}, std(alpha_hat) {mk['sigma_mock']:.4f} vs median "
          f"error {mk['sigma_med']:.4f}, pull std {mk['pull_std']:.2f}, 68% coverage {mk['coverage68']:.2f}")
    print(f"     marginal 68% intervals: coverage {mk['coverage68_marg']:.2f}, median std {mk['marg_std_med']:.4f}")
    print(f"  Sigma fit (marginal mean): bias {mkS['Sigma_bias']:+.2f}, scatter {mkS['Sigma_scatter']:.2f}, "
          f"median posterior std {mkS['Sigma_std_med']:.2f}, 68% coverage {mkS['Sigma_coverage68']:.2f}, "
          f"profile interval open in {mkS['Sigma_open']:.0%}")